# {{EXPERIMENT}} · LIBRARY: analysis (2 of 2)

<!-- describe the analysis here -->

Reads what `1-Process` wrote to `4_exports`; every filter is applied here. A step whose input is missing is skipped with a note.

## Configure

In [ ]:
EXPERIMENT = "{{EXPERIMENT}}"
SUBSTRATE = "bzp_uM"            # the assay table's [S] column (µM)
TAG_SUFFIX = None   # how figures write a TAG ("*") substitution: None "A4tag"; the amino acid incorporated, e.g. "AHA" (azidohomoalanine): "A4AHA"

# the BLANK floors (expression, buttons, diffusion): median + N_SD robust SDs (1.4826 x MAD; FLOOR = "sd": plain SDs)
N_SD, FLOOR = 3, "mad"

# the enzyme on the buttons: the scan (a `scan` label of gfp_per_chamber.csv; None: the last SDS wash) and the readout
# ("button_sum_net": the sum minus the ring inside the button rim, same scan)
BUTTON_SCAN = None
BUTTON_VALUE = "button_sum_net"

# tiles 1-Process's tile check flagged (a darker or brighter tile on one scan): buttons and chamber passes in them left out
DROP_FLAGGED_TILES = True

STANDARDS_CAMERA = None   # the NADH standards' camera folder (4_exports/assay/standards/<camera>); None: the only one
CONC_MAX = None           # calibrate on standards up to this [NADH] (µM)
RATE_RULES = {}           # overrides of hp.PROGRESS_4PL_RULES, e.g. {"min_rise_snr": 5}
MIN_R2 = 0.9              # Michaelis–Menten fits below this r² stay out of the report
MIN_REF_CHAMBERS = 3        # the volcanoes test against WT when at least this many WT chambers count, else the library median
REF_NEEDS_GFP_FLOOR = True   # False: every WT chamber with a passing fit is the reference, its button GFP above the floor or not


## Setup

In [ ]:
import htmek_processing as hp

an = hp.Analysis.from_config(globals())

# 1. Expression and enzyme

### 1a. Variants that did not express (in solution)

From the expression time course: a chamber expressed when the GFP it gains in its DNA chamber (background subtracted; last time point minus first) is above the BLANK floor; a variant did not express when none of its chambers did. The BLANKs give leakage from their neighbors and contamination. Report: `5_figures/qc/2_expression/expression_report.pdf`.

In [ ]:
an.call_expression()

### 1b. Enzyme on the buttons

GFP left on each button after the SDS washes, against the BLANK buttons' floor. Buttons in a tile flagged on that scan are left out (`DROP_FLAGGED_TILES`).

In [ ]:
an.call_buttons()

### 1c. Signal in the DNA chambers during the assay

The neck is closed during an assay, so the DNA chambers should stay flat; product diffusing under the neck makes them rise. Per chamber and run: the DNA chamber's slope against the BLANKs' floor, and its ratio to the reaction chamber's slope. Enzyme left in the DNA chamber after expression also makes it rise; a ratio above 1 can't be diffusion from the reaction chamber.

In [ ]:
an.check_diffusion()

# 2. NADH calibration

Per-chamber standard curves (chamber ROI median vs [NADH]); every pass converted to µM, and the rates then fit in µM with the [S] rules on. Skipped without standards: the rates stay in RFU.

In [ ]:
an.calibrate()

# 3. Photobleaching and initial rates

Bleaching modeled from the BLANKs per run (exponential loss to a floor, per exposure); then a 4PL and a line per chamber with `hp.PROGRESS_4PL_RULES`: v0 is the 4PL's steepest slope when every rule holds, else the line's. Complete passes only; chamber passes in a flagged tile left out.

In [ ]:
an.fit_rates()

**QC:** initial rate vs the enzyme on the button.

In [ ]:
an.rates_vs_enzyme()

# 4. Michaelis–Menten

v0 and v0 / [E] (buttons above the floor), one fit per chamber across the [S] runs. Needs ≥ 3 [S] runs.

In [ ]:
an.fit_mm()

# 5. Chamber report

One page per chamber: its button (the button GFP scan) and chamber (the highest [S], last pass), every run's progress curve and fit, and the MM fits raw and per GFP. Reads the stitched images on the drive. One copy in `5_figures/4_report`.

In [ ]:
an.chamber_report()

# 6. Experiment report

The chip-level picture: summary, chip maps, rate maps, trends, distributions, volcano statistics. One copy in `5_figures/4_report`.

In [ ]:
an.experiment_report()